In [1]:
import cv2
import json
import time
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from transformers import Qwen2TokenizerFast, Qwen3ForCausalLM, AutoConfig

from diffusers.utils import load_image
from diffusers.schedulers import FlowMatchEulerDiscreteScheduler
from diffusers.models import AutoencoderKLFlux2, Flux2Transformer2DModel

from ctgmworkshop.flux_tools import *
from ctgmworkshop.image_tools import *
from ctgmworkshop.optical_flow_tools import *
from ctgmworkshop.tiny_vae import DiffusersTAEF2Wrapper

device = "cuda"
dtype = torch.bfloat16
plt.style.use("dark_background")

In [2]:
text_encoder = Qwen3ForCausalLM.from_pretrained(
    "../../FLUX.2-klein-4B/text_encoder", local_files_only=True
).to(device, dtype)
tokenizer = Qwen2TokenizerFast.from_pretrained(
    "../../FLUX.2-klein-4B/tokenizer", local_files_only=True, device=device
)

scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
    "../../FLUX.2-klein-4B/scheduler", device=device
)
vae = AutoencoderKLFlux2.from_pretrained("../../FLUX.2-klein-4B/vae", device=device).to(
    device, dtype
)
transformer = Flux2Transformer2DModel.from_pretrained(
    "../../FLUX.2-klein-4B/transformer", device=device
).to(device, dtype)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
height = 512
width = 960

In [4]:
@torch.no_grad()
def process_frame(frame, seed, prompt):
    num_inference_steps = 2

    frame = norm_image(resize_torch(cv2_to_torch(frame), height=height, width=width))

    image_latents, image_latent_ids = prepare_image_latents(
        images=[frame], vae=vae, batch_size=1, device=device, dtype=dtype
    )

    cond_latents = unpack_latents_with_ids(image_latents, image_latent_ids)
    cond_latents = unpatchify_latents(cond_latents)

    # cond_latents -> return

    prompt_embeds, text_ids = encode_prompt(
        prompt=prompt,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        device=device,
        num_images_per_prompt=1,
        max_sequence_length=512,
        text_encoder_out_layers=(9, 18, 27),
    )

    generator = torch.Generator(device="cuda").manual_seed(seed)

    latents, latent_ids = prepare_latents(
        batch_size=1,
        num_latents_channels=32,
        height=height,
        width=width,
        dtype=dtype,
        device=device,
        generator=generator,
    )

    sigmas = np.linspace(1.0, 1 / num_inference_steps, num_inference_steps)

    image_seq_len = latents.shape[1]

    mu = compute_empirical_mu(
        image_seq_len=image_seq_len, num_steps=num_inference_steps
    )

    timesteps, num_inference_steps = retrieve_timesteps(
        scheduler,
        num_inference_steps,
        device,
        sigmas=sigmas,
        mu=mu,
    )

    scheduler.set_begin_index(0)

    init_latents = latents.clone()
    init_latents = unpack_latents_with_ids(init_latents, latent_ids)
    init_latents = unpatchify_latents(init_latents)

    # init_latents -> return

    for i, t in enumerate(timesteps):
        timestep = t.expand(latents.shape[0]).to(latents.dtype)

        latent_model_input = latents.to(transformer.dtype)
        latent_image_ids = latent_ids

        noise_pred = transformer(
            hidden_states=torch.cat([latent_model_input, image_latents], dim=1),
            timestep=timestep / 1000,
            guidance=None,
            encoder_hidden_states=prompt_embeds,
            txt_ids=text_ids,
            img_ids=torch.cat([latent_image_ids, image_latent_ids], dim=1),
            return_dict=False,
        )[0]

        noise_pred = noise_pred[:, : latents.size(1), :]

        latents = scheduler.step(noise_pred, t, latents, return_dict=False)[0]

    vae_scale_factor = 8

    latent_height = 2 * (int(height) // (vae_scale_factor * 2))
    latent_width = 2 * (int(width) // (vae_scale_factor * 2))

    latents = unpack_latents_with_ids(
        latents, latent_ids, latent_height // 2, latent_width // 2
    )

    final_latents = unpatchify_latents(latents)

    # final_latents -> return

    return cond_latents, init_latents, final_latents


In [5]:
def prepare_reference_latents(
    reference_latents: list[torch.Tensor],
    vae,
    batch_size,
    device,
    dtype,
):
    """Patchify and pack a list of unpatchified reference latents into a single
    sequence for use as image conditioning tokens in the Flux transformer."""
    image_latents = []
    for l in reference_latents:
        image_latent = patchify_latents(l)
        image_latents.append(image_latent)

    image_latent_ids = prepare_image_ids(image_latents)

    packed_latents = []
    for latent in image_latents:
        packed = pack_latents(latent)  # (1, seq, C)
        packed = packed.squeeze(0)  # (seq, C)
        packed_latents.append(packed)

    image_latents = torch.cat(packed_latents, dim=0)  # (N*seq, C)
    image_latents = image_latents.unsqueeze(0)  # (1, N*seq, C)

    image_latents = image_latents.repeat(batch_size, 1, 1)
    image_latent_ids = image_latent_ids.repeat(batch_size, 1, 1)
    image_latent_ids = image_latent_ids.to(device)

    return image_latents, image_latent_ids

In [11]:
@torch.no_grad()
def refract(cond_A, cond_B, final_A, prompt):
    full_prompt = "Fill gray masked regions. " + prompt

    prompt_embeds, text_ids = encode_prompt(
        prompt=full_prompt,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        device=device,
        num_images_per_prompt=1,
        max_sequence_length=512,
        text_encoder_out_layers=(9, 18, 27),
    )

    generator = torch.Generator(device="cuda").manual_seed(67)

    latents, latent_ids = prepare_latents(
        batch_size=1,
        num_latents_channels=32,
        height=height,
        width=width,
        dtype=dtype,
        device=device,
        generator=generator,
    )

    image_latents, image_latent_ids = prepare_reference_latents(
        reference_latents=[cond_B, final_A],
        vae=vae,
        batch_size=1,
        device=device,
        dtype=dtype,
    )
    image_latents = image_latents.to(transformer.dtype)

    num_inference_steps = 4
    sigmas = np.linspace(1.0, 1 / num_inference_steps, num_inference_steps)
    image_seq_len = latents.shape[1]
    mu = compute_empirical_mu(
        image_seq_len=image_seq_len, num_steps=num_inference_steps
    )

    timesteps, num_inference_steps = retrieve_timesteps(
        scheduler,
        num_inference_steps,
        device,
        sigmas=sigmas,
        mu=mu,
    )

    for i, t in enumerate(timesteps):
        timestep = t.expand(latents.shape[0]).to(latents.dtype)

        noise_pred = transformer(
            hidden_states=torch.cat(
                [latents.to(transformer.dtype), image_latents], dim=1
            ),
            timestep=timestep / 1000,
            guidance=None,
            encoder_hidden_states=prompt_embeds,
            txt_ids=text_ids,
            img_ids=torch.cat([latent_ids, image_latent_ids], dim=1),
            return_dict=False,
        )[0]

        noise_pred = noise_pred[:, : latents.size(1), :]
        latents = scheduler.step(noise_pred, t, latents, return_dict=False)[0]

    latents = unpack_latents_with_ids(latents, latent_ids)
    latents = unpatchify_latents(latents)

    return latents

In [12]:
taef2_diffusers = (
    DiffusersTAEF2Wrapper(path="../../taef2.safetensors").eval().requires_grad_(False)
)
tiny_vae = taef2_diffusers.to(device, dtype)

In [20]:
prompts = [
    "Turn this into art in style of Kandinsky",
    "Turn this into impressionism art with many details",
    "Turn this into cyberpunk city, neon lamps and signs, night, sharp details, reflections on the wet ground",
    "All people are now wearing red shirts",
    "Make it winter",
    "Turn this into post apocaliptic city street, buildings and roads are covered with moss and vegetation, sunny, beautiful",
]


def build_random_prompt():
    return random.choice(prompts)

In [21]:
raft_model, raft_preprocess = get_large_raft()

In [22]:
cap = cv2.VideoCapture("../../local_samples/walking.mp4")

skip = 20500

for i in range(skip):
    ret, frame = cap.read()


fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # Codec for MP4
out = cv2.VideoWriter("output_video_3.mp4", fourcc, 30, (width, height))


while True:
    current_prompt = build_random_prompt()
    cond_latents, _, final_latents = process_frame(
        frame, random.randint(0, 10000), current_prompt
    )
    frame_A = cv2_to_np(resize_cv2(frame, height=height, width=width))

    interrupted = False
    for i in range(30):
        ret, frame = cap.read()
        frame_B = cv2_to_np(resize_cv2(frame, height=height, width=width))
        if not ret:
            break

        current_cond = cv2_to_np(resize_cv2(frame, height=height, width=width))
        current_cond = np_to_torch(current_cond)
        current_cond = norm_image(current_cond)
        current_cond, ids = prepare_image_latents(
            [current_cond], tiny_vae, batch_size=1, device=device, dtype=dtype
        )
        current_cond = unpack_latents_with_ids(current_cond, ids)
        current_cond = unpatchify_latents(current_cond)
        current_cond = current_cond.float()

        # mask overlaping

        flow_AB = flow_from_numpy(
            frame_A,
            frame_B,
            model=raft_model,
            preprocess=raft_preprocess,
        )
        flow_BA = flow_from_numpy(
            frame_B,
            frame_A,
            model=raft_model,
            preprocess=raft_preprocess,
        )
        mask = bidir_flow_mask(flow_AB, flow_BA, threshold=10.0)

        processed_B_cond = decode_latents(final_latents, tiny_vae)
        processed_B_cond = warp_image(processed_B_cond.float(), flow_AB.float()).to(
            dtype
        )
        processed_B_cond = processed_B_cond * mask
        processed_B_cond = encode_latents(processed_B_cond, tiny_vae)

        final_image_A = torch_to_cv2(
            denorm_image(decode_latents(processed_B_cond, tiny_vae))
        )
        final_image_A = resize_cv2(final_image_A, height=128, width=240)

        final_B = refract(
            cond_A=cond_latents,
            cond_B=current_cond,
            final_A=processed_B_cond,
            prompt=current_prompt,
        )

        final_image_B = torch_to_cv2(denorm_image(decode_latents(final_B, tiny_vae)))

        out_frame = final_image_B

        out_frame[:128, :240, :] = resize_cv2(frame, height=128, width=240)
        out_frame[:128, -240:, :] = final_image_A

        cv2.imshow("Video", out_frame)
        out.write(out_frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            interrupted = True
            break

    if interrupted:
        break


cap.release()
out.release()
cv2.destroyAllWindows()